# TriPendulum-8: Goal-Conditioned RL for Cart-Triple Pendulum

This notebook provides the complete pipeline for setting up, training, and evaluating the **TriPendulum-8** controller. The project uses MuJoCo, Gymnasium, and Stable-Baselines3 (SAC/PPO) to achieve stable goal-directed behavior across 8 absolute target poses under limited track boundaries.

## 1. Setup & Installation
Install the required packages. In Google Colab, this automatically configures headless MuJoCo rendering.

In [ ]:
# Install package in editable mode with dependencies
!pip install -e . 
!pip install imageio-ffmpeg tabulate pandas seaborn opencv-python

## 2. Environment Verification
Verify that the MuJoCo engine can load the XML and create the `TriPendulumGoalEnv` correctly.

In [ ]:
import gymnasium as gym
import numpy as np
import envs

# Create the environment
env = gym.make("TriPendulumGoalEnv-v0")
obs, info = env.reset()

print("Environment created successfully!")
print("Observation space shape:", env.observation_space.shape)
print("Action space:", env.action_space)
print("Initial Observation:", obs)
print("Initial Info:", info)

## 3. Verify Relative-to-Absolute Coordinate Conversion & 8 Goals
Print the absolute target angles and verify the angle wrap mechanics.

In [ ]:
from envs.goals import GOAL_NAMES, GOAL_ABS_ANGLES, get_goal
from utils.angle_utils import relative_to_absolute, wrap_angle

print("8 Goal Poses absolute target angles (in degrees):\n")
for goal in GOAL_NAMES:
    angles = GOAL_ABS_ANGLES[goal]
    deg_angles = np.degrees(angles)
    print(f"{goal:<4} -> Absolute Angles: {deg_angles} deg")

print("\nTesting coordinate mapping:")
q_rel = [np.pi/4, np.pi/4, np.pi/4] # relative joint angles
theta_abs = relative_to_absolute(q_rel)
print("Relative joints q:", q_rel)
print("Computed Absolute angles theta:", theta_abs)
print("Wrapped absolute angles:", wrap_angle(theta_abs))

## 4. Run Random Action Test Rollouts
Run the environment with random controls to check for physical boundary hits and state responses.

In [ ]:
obs, info = env.reset(goal="UUU")
done = False
truncated = False
steps = 0

print("Running random action loop until termination...")
while not (done or truncated):
    action = env.action_space.sample()
    obs, reward, done, truncated, info = env.step(action)
    steps += 1
    if steps % 10 == 0:
        print(f"Step: {steps:<3} | Cart X: {info['x']:+0.3f}m | Max X: {info['max_abs_x']:.3f}m | Reward: {reward:+.3f} | Collided: {info['track_collision']}")
        
print(f"\nRollout ended after {steps} steps. Info:", info)

## 5. Train baseline PPO controller
Run PPO baseline training. In Colab, we can set `--total-timesteps` to train for a shorter or longer duration.

In [ ]:
# Train PPO for 100,000 steps
!python training/train_ppo.py --total-timesteps 100000 --tb-log runs/ppo

## 6. Train main SAC controller (Curriculum Enabled)
Train the primary SAC model. Checkpoints and evaluation reports will be updated live.

In [ ]:
# Train SAC for 200,000 steps
!python training/train_sac.py --total-timesteps 200000 --tb-log runs/sac

## 7. Open TensorBoard
Start Tensorboard to track rewards and losses live.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 8. View Training Analytics Reports
View the latest evaluation report compiled by the custom analytics manager.

In [ ]:
import os
import glob
from IPython.display import Markdown, display

reports = sorted(glob.glob("results/analytics/step_*/report_step_*.md"))
if reports:
    print(f"Displaying latest report: {reports[-1]}")
    with open(reports[-1], "r") as f:
        display(Markdown(f.read()))
else:
    print("No step reports generated yet. Training must run past the first eval_freq step.")

## 9. Play Worst-Goal Diagnostics Videos
Embed and play the generated MP4 videos for the worst-performing goals inside the notebook cell.

In [ ]:
import io
import base64
from IPython.display import HTML

videos = sorted(glob.glob("videos/diagnostics/*.mp4"))
if videos:
    video_path = videos[-1]
    print(f"Playing diagnostic video: {video_path}")
    
    video_file = io.open(video_path, 'r+b').read()
    encoded = base64.b64encode(video_file)
    display(HTML(data='''<video alt="test" autoplay 
                loop controls style="height: 360px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
else:
    print("No diagnostic videos found.")

## 10. Quantitative Evaluation & Transition Matrix
Run evaluation scripts to compute success statistics and transition properties, then render heatmaps.

In [ ]:
# Run comprehensive evaluation
!python evaluation/evaluate.py --model checkpoints/sac_best.zip --episodes 10

# Evaluate transitions (8x8 = 64 pairs)
!python evaluation/transition_matrix.py --model checkpoints/sac_best.zip --trials 2

# Refresh dashboard charts
!python dashboard/plots.py

## 11. Live Dashboard Panel
Renders the interactive LivePanel in Colab, refreshing plots and warning boxes.

In [ ]:
from dashboard.live_panel import LivePanel

panel = LivePanel()
# Update and render the live panel
panel.update()